# 02 · Anonimización determinística (sin IA)

<a href="https://colab.research.google.com/github/manuelarguelles/tyv-demo-colab/blob/main/notebooks/02_anonimizacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

Segundo paso del pipeline: antes de que **cualquier** modelo de lenguaje vea
el currículum, el sistema retira los datos que identifican a la persona
(nombre, correo, teléfono, DNI, fecha de nacimiento, dirección) y los
reemplaza por marcadores como `[CORREO_1]`.

**Punto clave, y la razón de que este paso tenga su propio notebook:**
esto **no lo decide un modelo de IA**. Es una transformación 100%
determinística por **expresiones regulares** en Python — el mismo texto de
entrada siempre produce exactamente la misma salida, sin variabilidad ni
"criterio" de ningún modelo. Eso es lo que la hace auditable.

**Límite declarado, no escondido:** anonimizar por patrones deja pasar lo
que no tiene forma de patrón reconocible. Es una capa de protección, no una
garantía absoluta de anonimato — la literatura (Parasurama y Sedoc, 2022)
muestra que el propio lenguaje de un CV puede filtrar señales como el
género incluso sin nombres.


## 1. El texto de entrada (ya extraído del PDF en el notebook anterior)

In [ ]:
CV_TEXTO = """ALEJANDRA ROJAS MEDINA
Lima, Perú · a.rojas.medina@ejemplo.com · +51 987 654 321
Jr. Los Alamos 245, San Isidro, Lima

FORMACIÓN ACADÉMICA
Bachiller en Derecho — Universidad Nacional Mayor de San Marcos (2015 – 2020)
Diplomado en Derecho Laboral — Pontificia Universidad Católica del Perú (2021)
Certificación en Protección de Datos Personales — Indecopi (2022)

EXPERIENCIA PROFESIONAL
Asistente Legal Junior — Estudio Fernández & Asociados (2020 – 2022)
  Apoyo en la elaboración de contratos laborales y absolución de consultas
  de clientes corporativos sobre normativa de protección de datos.

Analista Legal — Grupo Andino S.A.C. (2022 – Presente)
  Responsable de la revisión de políticas internas de privacidad y de la
  coordinación con el área de Recursos Humanos en procesos disciplinarios.

CONOCIMIENTOS TÉCNICOS
Manejo de bases de datos jurisprudenciales (LP, Actualidad Jurídica).
Redacción de informes legales y absolución de consultas escritas.
Nivel intermedio de inglés (certificado ICPNA).

Fecha de nacimiento: 14 de marzo de 1994
DNI: 45678912"""
print(CV_TEXTO)


## 2. Los patrones de anonimización

Cada patrón se ejecuta **en orden**, y el orden es parte del diseño:

1. **Correo** primero — porque un correo puede contener dígitos que el patrón
   de teléfono o DNI reconocería; redactarlo a medias dejaría el dominio a
   la vista.
2. **Teléfono** (móvil peruano de 9 dígitos empezando en 9, o fijo de Lima
   con prefijo 01) — exigiendo el prefijo completo para no confundir un
   rango de años ("2019 – 2026") con un número de teléfono.
3. **Documento de identidad** (DNI de 8 dígitos, o carné de extranjería /
   pasaporte).
4. **Fecha de nacimiento**.
5. **Dirección** (avenida, jirón, calle, urbanización...).
6. **Nombre** — al final, porque un nombre no tiene "forma" de patrón: solo
   se puede reconocer si el sistema ya lo conoce (viene del formulario de
   postulación), y porque redactarlo antes que el correo partiría un correo
   como `maria.rojas@estudio.pe` en `maria.[NOMBRE_1]@estudio.pe`, dejando
   el dominio sin redactar.


In [ ]:
import re
from dataclasses import dataclass
from typing import Sequence

@dataclass(frozen=True)
class Redaccion:
    tipo: str
    original: str
    marcador: str

_SEP = r"[\s.\x1f-]"  # separadores que la extracción de PDF deja entre dígitos

PATRONES = (
    ("correo", re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")),
    # Móvil peruano: 9 dígitos empezando en 9, con o sin +51, separadores repetidos.
    ("telefono", re.compile(rf"(?:\+?51{_SEP}{{0,3}})?9(?:{_SEP}{{0,3}}\d){{8}}(?!\d)")),
    # Fijo de Lima: prefijo 01 completo (no un "1" suelto, para no confundir años).
    ("telefono", re.compile(rf"\(?\s*0{_SEP}{{0,2}}1\s*\)?{_SEP}{{0,3}}\d(?:{_SEP}{{0,3}}\d){{6}}(?!\d)")),
    ("documento", re.compile(r"\b\d{8}\b")),
    ("documento", re.compile(r"\b(?:CE|C\.E\.|pasaporte)[\s:]*[A-Z0-9]{6,12}\b", re.IGNORECASE)),
    ("fecha_de_nacimiento", re.compile(
        r"\b(?:fecha\s+de\s+nacimiento|nacid[oa]\s+el|f\.?\s?nac\.?)[\s:]*\d{1,2}[/\-\s]\w{1,10}[/\-\s]\d{2,4}",
        re.IGNORECASE)),
    ("direccion", re.compile(
        r"\b(?:av\.?|avenida|jr\.?|jir[oó]n|calle|urb\.?|urbanizaci[oó]n|mz\.?|psje\.?|pasaje)\s+[^\n,;]{3,60}",
        re.IGNORECASE)),
)
print(f"{len(PATRONES)} patrones cargados.")


## 3. La función de anonimización

In [ ]:
def anonimizar(texto: str, nombres_conocidos: Sequence[str] = ()) -> tuple[str, list[Redaccion]]:
    """Aplica los patrones en orden y luego los nombres conocidos.
    Devuelve (texto_protegido, lista_de_redacciones) — la lista es el
    registro auditable de qué se redactó y con qué marcador."""
    redacciones: list[Redaccion] = []
    resultado = texto
    contadores: dict[str, int] = {}

    def marcador_de(tipo: str) -> str:
        contadores[tipo] = contadores.get(tipo, 0) + 1
        # El marcador deja ver QUE había un dato y de qué tipo -- borrarlo del
        # todo dejaría al evaluador leyendo una frase rota.
        return f"[{tipo.upper()}_{contadores[tipo]}]"

    for tipo, expresion in PATRONES:
        def reemplazo(m: "re.Match[str]", _tipo=tipo) -> str:
            marcador = marcador_de(_tipo)
            redacciones.append(Redaccion(tipo=_tipo, original=m.group(0), marcador=marcador))
            return marcador
        resultado = expresion.sub(reemplazo, resultado)

    # Nombres: van AL FINAL (ver explicación arriba), completos y por partes
    # sueltas (una firma puede decir solo "Rojas").
    for nombre in nombres_conocidos:
        partes = [p.strip() for p in re.split(r"\s+", nombre) if len(p.strip()) >= 3]
        candidatos = sorted({nombre, *partes}, key=len, reverse=True)
        for aguja in candidatos:
            expresion = re.compile(rf"\b{re.escape(aguja)}\b", re.IGNORECASE)
            def reemplazo_nombre(m: "re.Match[str]") -> str:
                marcador = marcador_de("nombre")
                redacciones.append(Redaccion(tipo="nombre", original=m.group(0), marcador=marcador))
                return marcador
            resultado = expresion.sub(reemplazo_nombre, resultado)

    return resultado, redacciones


## 4. Aplicarla sobre el CV de ejemplo

Le pasamos el nombre conocido (viene del formulario de postulación, no del propio CV — el sistema real lo recibe así).

In [ ]:
cv_protegido, redacciones = anonimizar(CV_TEXTO, nombres_conocidos=["Alejandra Rojas Medina"])

print(cv_protegido)


## 5. El registro auditable de redacciones

In [ ]:
for r in redacciones:
    print(f"{r.tipo:20s} {r.marcador:14s} ← {r.original!r}")

print(f"\nTotal: {len(redacciones)} dato(s) personal(es) redactado(s).")


## 6. Verificación: ¿quedó algo sin redactar?

Un oráculo *independiente* de la propia función de anonimización — usa las
mismas formas de detección, pero sirve para auditar el resultado, no para
generar el reemplazo (si usara la misma lógica que genera, un bug en el
patrón nunca se detectaría a sí mismo).

In [ ]:
def filtraciones_identificatorias(texto: str) -> list[str]:
    sospechas = []
    for expresion, etiqueta in (
        (re.compile(r"[\w.+-]+@[\w-]+\.\w{2,}"), "correo"),
        (re.compile(r"\b\d{8}\b"), "ocho dígitos seguidos"),
    ):
        for m in expresion.finditer(texto):
            sospechas.append(f"{etiqueta}: {m.group(0)}")
    return sospechas

fugas = filtraciones_identificatorias(cv_protegido)
print("Sin fugas detectadas." if not fugas else f"⚠ Posibles fugas: {fugas}")


## Siguiente paso

`cv_protegido` es el texto que sí puede llegar a un modelo de lenguaje. En
`03_siete_consultas_llm.ipynb` lo evaluamos contra los 7 criterios de la
rúbrica, uno por uno.

---
*Este material es contenido educativo de apoyo a una tesis de maestría (Terry & Valdez — sistema de filtrado curricular). El CV usado es 100% ficticio, construido para esta demostración. Ningún dato de candidatos reales del proyecto se publica en este repositorio: ver `materiales/README.md` para trabajar con datos reales de forma local.*
